In [1]:
import sys
from pathlib import Path
try:
	base_dir = Path(__file__).resolve().parent
except NameError:
	# Jupyter Notebook 环境下没有 __file__
	base_dir = Path.cwd().resolve().parent

sys.path.append(str(base_dir))

import pandas as pd
import numpy as np
import json
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from pipeline.tfclf import makePipe
from tqdm import tqdm, trange

DATA_PATH = "../data/train.csv"
SPLIT_PATH = "../data/split_indices.json"
TRAIN_SPLIT_NAME = "train_ids"
DEV_SPLIT_NAME = "dev_ids"
TEST_SPLIT_NAME = "heldout_ids"

# Load Data

In [2]:
raw_dataset = pd.read_csv(DATA_PATH)
with open(SPLIT_PATH, 'r') as f :
	raw_split : dict = json.load(f)

dev_ids = raw_split['dev_ids']
heldout_ids = raw_split['heldout_ids']

de_dataset = raw_dataset[raw_dataset['id'].isin(raw_split[DEV_SPLIT_NAME])]
tr_dataset = raw_dataset[raw_dataset['id'].isin(raw_split[TRAIN_SPLIT_NAME])]
te_dataset = raw_dataset[raw_dataset['id'].isin(raw_split[TEST_SPLIT_NAME])]

de_ids = de_dataset['id']
tr_ids = tr_dataset['id']
te_ids = te_dataset['id']

print(f"dev items: {len(de_dataset)} \ttrain items: {len(tr_dataset)} \theldout items: {len(te_dataset)}")
# mutual check
if de_ids.isin(tr_ids).sum() > 0 or de_ids.isin(te_ids).sum() > 0 or tr_ids.isin(te_ids).sum() > 0 :
	print(f"Mutual check is failed")
else :
	print(f"Mutual check is successful")

de_dataset

dev items: 1523 	train items: 4567 	heldout items: 1523
Mutual check is successful


,id,keyword,location,text,target
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
10,16,NaN,NaN,Three people died from the heat wave so far,1
12,18,NaN,NaN,#raining #flooding #Florida #TampaBay #Tampa 1...,1
14,20,NaN,NaN,Damage to school bus on 80 in multi car crash ...,1
...,...,...,...,...,...
7579,10831,wrecked,"Vancouver, Canada",Three days off from work and they've pretty mu...,0
7583,10835,NaN,NaN,Pic of 16yr old PKK suicide bomber who detonat...,1
7595,10850,NaN,NaN,NWS: Flash Flood Warning Continued for Shelby ...,1
7598,10853,NaN,NaN,Father-of-three Lost Control of Car After Over...,1


# Baselines

## Floor Model

Prediction the most frequent target label in training split

In [3]:
floor_tgt, mx_num = None, 0
for tgt, grp in tr_dataset.groupby("target") :
	if len(grp) > mx_num :
		floor_tgt, mx_num = tgt, len(grp)

print(f"Floor Model Prediction: {floor_tgt}")
floor_pred = pd.Series([floor_tgt] * len(te_dataset))
print(f"Floor Model Accuracy: {accuracy_score(te_dataset['target'], floor_pred)}")

Floor Model Prediction: 0
Floor Model Accuracy: 0.5705843729481287


## TF-IDF + Logistic Regression

In [9]:
pipe = makePipe({"clf__max_iter": 3000})

param_grid = {
    # TF-IDF 参数
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3, 5],
    "tfidf__max_df": [0.9, 0.95, 1.0],
    "tfidf__max_features": [20000, 30000, 50000, None],

    # Logistic Regression 参数
    "clf__C": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "clf__solver": ["saga"],
    # "clf__penalty": ["l2"],
    "clf__class_weight": [None, "balanced"],
}

grid = GridSearchCV(
	estimator=pipe,
	param_grid=param_grid,
	scoring='f1',
	cv=5,
	n_jobs=-1,
)

grid.fit(de_dataset['text'], de_dataset['target'])

print("Best CV score on dev split: ", grid.best_score_)
print("Best params: ", grid.best_params_)

bst_pipe = makePipe(grid.best_params_)

bst_pipe.fit(tr_dataset['text'], tr_dataset['target'])

te_pred = bst_pipe.predict(te_dataset['text'])

print(f"F1 score: {f1_score(te_dataset['target'], te_pred)}")
print(f"Accuracy: {accuracy_score(te_dataset['target'], te_pred)}")

Best CV score on dev split:  0.6645361486570893
Best params:  {'clf__C': 2.0, 'clf__class_weight': 'balanced', 'clf__solver': 'saga', 'tfidf__max_df': 0.9, 'tfidf__max_features': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}
F1 score: 0.7593014426727411
Accuracy: 0.7918581746552856
